# Week 4 Capstone: End-to-End Customer Churn Prediction

## Executive Summary & Problem Statement
Customer attrition (churn) directly reduces recurring revenues and increases acquisition costs in subscription and telecom services. The goal of this capstone project is to develop, evaluate, and serialize an end-to-end Machine Learning pipeline that identifies customers at risk of churning.

### Project Flow
1. **Data Ingestion & Inspection**
2. **Exploratory Data Analysis (EDA)**
3. **Preprocessing & Feature Engineering Pipeline**
4. **Supervised Model Training & Cross-Comparison**
5. **Hyperparameter Tuning (GridSearchCV)**
6. **Evaluation (Accuracy, Precision, Recall, F1, ROC-AUC)**
7. **Model Serialization (`joblib.dump`) for REST API Deployment**

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import joblib
from pathlib import Path

from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.impute import SimpleImputer
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score, 
    roc_auc_score, roc_curve, confusion_matrix, classification_report
)

# Set plotting style
sns.set_theme(style='whitegrid', palette='muted')
plt.rcParams['figure.figsize'] = (10, 6)

## 1. Load Dataset
Loading the customer dataset from `../data/dataset.csv`.

In [ ]:
data_path = Path('../data/dataset.csv')
df = pd.read_csv(data_path)

print(f'Dataset Shape: {df.shape}')
df.head()

In [ ]:
# Dataset info and data types
df.info()
print('\nMissing values per column:\n', df.isnull().sum())

## 2. Exploratory Data Analysis (EDA)
Examining target distribution, tenure vs churn, and monthly charges.

In [ ]:
plt.figure(figsize=(6, 4))
sns.countplot(data=df, x='Churn')
plt.title('Customer Churn Class Distribution')
plt.xlabel('Churn Status')
plt.ylabel('Count')
plt.show()

churn_rates = df['Churn'].value_counts(normalize=True) * 100
print(f'Churn Rate:\n{churn_rates.round(2)}%')

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
sns.boxplot(data=df, x='Churn', y='MonthlyCharges', ax=axes[0])
axes[0].set_title('Monthly Charges vs Churn')

sns.boxplot(data=df, x='Churn', y='tenure', ax=axes[1])
axes[1].set_title('Tenure (Months) vs Churn')
plt.tight_layout()
plt.show()

## 3. Preprocessing & Pipeline Architecture
To prevent data leakage and make the model ready for a REST API, we construct an encapsulated `ColumnTransformer` with:
- **Numerical features**: Median Imputation + `StandardScaler`
- **Categorical features**: Most Frequent Imputation + `OneHotEncoder(handle_unknown='ignore')`

In [ ]:
# Separate identifiers, features, and target
if 'customerID' in df.columns:
    df = df.drop(columns=['customerID'])

df['TotalCharges'] = pd.to_numeric(df['TotalCharges'], errors='coerce')
y = df['Churn'].map({'Yes': 1, 'No': 0})
X = df.drop(columns=['Churn'])

numerical_cols = ['tenure', 'MonthlyCharges', 'TotalCharges']
categorical_cols = [c for c in X.columns if c not in numerical_cols]

# Train-test split with stratification
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

print(f'Train samples: {X_train.shape[0]}, Test samples: {X_test.shape[0]}')

num_pipe = Pipeline([
    ('imputer', SimpleImputer(strategy='median')),
    ('scaler', StandardScaler())
])

cat_pipe = Pipeline([
    ('imputer', SimpleImputer(strategy='most_frequent')),
    ('ohe', OneHotEncoder(handle_unknown='ignore', sparse_output=False))
])

preprocessor = ColumnTransformer([
    ('num', num_pipe, numerical_cols),
    ('cat', cat_pipe, categorical_cols)
])

## 4. Supervised Model Training & Cross-Comparison
Comparing 4 candidate classifiers:
1. Logistic Regression
2. Decision Tree Classifier
3. Random Forest Classifier
4. Gradient Boosting Classifier

In [ ]:
candidate_models = {
    'Logistic Regression': LogisticRegression(max_iter=1000, random_state=42),
    'Decision Tree': DecisionTreeClassifier(max_depth=6, random_state=42),
    'Random Forest': RandomForestClassifier(n_estimators=150, max_depth=8, random_state=42),
    'Gradient Boosting': GradientBoostingClassifier(n_estimators=120, learning_rate=0.08, max_depth=4, random_state=42)
}

eval_results = []
trained_models = {}

for name, model in candidate_models.items():
    pipe = Pipeline([
        ('preprocessor', preprocessor),
        ('classifier', model)
    ])
    pipe.fit(X_train, y_train)
    trained_models[name] = pipe
    
    y_pred = pipe.predict(X_test)
    y_proba = pipe.predict_proba(X_test)[:, 1]
    
    eval_results.append({
        'Model': name,
        'Accuracy': round(accuracy_score(y_test, y_pred), 4),
        'Precision': round(precision_score(y_test, y_pred, zero_division=0), 4),
        'Recall': round(recall_score(y_test, y_pred, zero_division=0), 4),
        'F1 Score': round(f1_score(y_test, y_pred, zero_division=0), 4),
        'ROC-AUC': round(roc_auc_score(y_test, y_proba), 4)
    })

comparison_df = pd.DataFrame(eval_results).sort_values(by='F1 Score', ascending=False)
display(comparison_df)

## 5. ROC-AUC Curves and Confusion Matrix Analysis

In [ ]:
plt.figure(figsize=(8, 6))
for name, pipe in trained_models.items():
    y_proba = pipe.predict_proba(X_test)[:, 1]
    fpr, tpr, _ = roc_curve(y_test, y_proba)
    auc = roc_auc_score(y_test, y_proba)
    plt.plot(fpr, tpr, label=f'{name} (AUC = {auc:.3f})')

plt.plot([0, 1], [0, 1], 'k--', label='Chance (AUC = 0.50)')
plt.xlabel('False Positive Rate')
plt.ylabel('True Positive Rate')
plt.title('ROC Curves - Model Comparison')
plt.legend(loc='lower right')
plt.show()

## 6. Best Model Selection & Hyperparameter Optimization
We select the best-performing ensemble model (Random Forest / Gradient Boosting) and tune hyperparameters using `GridSearchCV`.

In [ ]:
param_grid = {
    'classifier__n_estimators': [100, 200],
    'classifier__max_depth': [6, 10, None],
    'classifier__min_samples_split': [2, 5]
}

base_pipe = Pipeline([
    ('preprocessor', preprocessor),
    ('classifier', RandomForestClassifier(random_state=42))
])

grid_search = GridSearchCV(
    base_pipe, param_grid, cv=3, scoring='f1', n_jobs=-1, verbose=1
)
grid_search.fit(X_train, y_train)

print('Best Hyperparameters:', grid_search.best_params_)
final_model = grid_search.best_estimator_

# Final evaluation on test set
y_final_pred = final_model.predict(X_test)
print('\nFinal Classification Report:\n')
print(classification_report(y_test, y_final_pred, target_names=['No Churn', 'Churn']))

## 7. Model Serialization with Joblib
We serialize the complete `final_model` pipeline (containing both preprocessing transformer and trained estimator) to `../model/model.pkl`.

This allows our API (`app.py`) to consume raw input JSON dictionaries without needing any custom external feature transformation code.

In [ ]:
model_dir = Path('../model')
model_dir.mkdir(parents=True, exist_ok=True)
model_export_path = model_dir / 'model.pkl'

joblib.dump(final_model, model_export_path)
print(f'Model pipeline successfully serialized to: {model_export_path}')

# Verify deserialization
loaded_pipeline = joblib.load(model_export_path)
test_sample = X_test.iloc[0:1]
pred = loaded_pipeline.predict(test_sample)[0]
prob = loaded_pipeline.predict_proba(test_sample)[0][1]

print(f'Deserialization test passed!')
print(f'Test Input: {test_sample.to_dict(orient="records")[0]}')
print(f'Predicted Class: {pred} (Churn Probability: {prob:.4f})')